# Brain MRI · Pretrained baseline B0

Một sample → 12 ảnh MRI thực → **MedGemma1.5 4B** → observation → bounded T2 edits → presenter.
Baseline này chỉ chỉnh primary-finding T2 direction trong draft được cung cấp.
Notebook không tự sinh report mới; draft lấy trong sample. `generated08` là cached draft có provenance.

**Chạy mới:** Colab GPU, upload source ZIP + một sample ZIP vào Drive `research/brain_vlm_xai/experiment/`.
**Đọc lại:** đặt `MODE="read"`, nhập `READ_RUN`; không cần GPU/weights/sample, vẫn cần source ZIP chứa reader.
Các code cells được đánh số 1–6. Sau disconnect, chạy lại cùng config và `RESUME_RUN` chỉ với source mới này.
Không resume run của notebook cũ dưới source đã refactor. Không có model fallback giả.

## 1 · Cấu hình và Drive

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.util, json, os, subprocess, sys, zipfile
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = str
    def display(value): print("No notebook display; open the printed presenter path in a browser.")

MODE = "run"                      # "run" = real inference; "read" = saved result only
SAMPLE_NAME = "sample_001"          # sample_001 … sample_004, all MEN development cases
DRAFT_VARIANTS = ["source", "controlled_low"]
RESUME_RUN = ""                    # only a matching run made with this source
READ_RUN = ""                      # required when MODE="read"
INSTALL_DEPS = True
DOWNLOAD_MISSING = True
SLICES_PER_SEQUENCE = 3

if MODE not in {"run", "read"}: raise ValueError("MODE must be run or read")
ON_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/research/brain_vlm_xai")
    PACKAGE_DIR = PROJECT / "experiment"
    CACHE = Path("/content/brain_mri_baseline_clean_v1")
    MODEL_ROOT = PROJECT / "assessor_models/medgemma-1.5-4b-it"
    ENV_ROOT = Path("/content/brain_vlm_envs/medgemma")
else:
    PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"src/brain_vlm").is_dir())
    PACKAGE_DIR = PROJECT / "releases/baseline_01"
    CACHE = PROJECT / "outputs/.tmp/baseline_clean_v1"
    MODEL_ROOT = PROJECT / "models/medgemma-1.5-4b-it"
    ENV_ROOT = PROJECT / ".venv_medgemma"
if MODE == "read" and not READ_RUN.strip(): raise ValueError("Set READ_RUN to the saved run folder")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_ROOT = Path(READ_RUN) if MODE == "read" else (Path(RESUME_RUN) if RESUME_RUN else
    (PACKAGE_DIR/"runs" if ON_COLAB else PROJECT/"outputs/runs") / ("baseline_clean_"+stamp))
print("MODE:", MODE, "| run:", RUN_ROOT)

## 2 · Source, sample và môi trường

Kiểm hash ZIP, dùng subprocess tách biệt nên không reuse module `brain_vlm` cũ trong notebook kernel.
Chỉ mode run mới cài dependencies. Cell này chưa load weights. Logs nằm trong run/logs.

In [ ]:
def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""): h.update(block)
    return h.hexdigest()

def unpack(path, expected):
    if not path.is_file(): raise FileNotFoundError(f"Upload {path.name} to {PACKAGE_DIR}")
    digest = sha(path)
    if digest != expected: raise ValueError("ZIP hash mismatch: "+path.name)
    target = CACHE / (path.stem+"_"+digest[:12])
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for item in archive.infolist():
            if not (target/item.filename).resolve().is_relative_to(target.resolve()): raise ValueError("Unsafe ZIP path")
        archive.extractall(target)
    return target

CODE_ROOT = unpack(PACKAGE_DIR/"brain_mri_baseline_clean_v1.zip", "1ec15cbbf2cf7f367acb1ff9d1e8bf7283f9bd10a5db6a16457eecbb4cd8834e")
manifest = json.loads((CODE_ROOT/"manifest.json").read_text())
for name, digest in manifest["files"].items():
    target = (CODE_ROOT/name).resolve()
    if not target.is_relative_to(CODE_ROOT.resolve()) or sha(target) != digest: raise ValueError("Changed source: "+name)
ENV = dict(os.environ, PYTHONPATH=str(CODE_ROOT/"src"), PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8", USE_TF="0", USE_FLAX="0")

def command(args):
    subprocess.run([str(a) for a in args], env=ENV, check=True)

if MODE == "run":
    SAMPLE_HASHES = {'sample_001': 'c1e7eb45911e9c767822944603f55c80d6b296fc9ac4d33676d78d9c55ac2c05', 'sample_002': '4f27619d405c1c36923b9a603b454a6860ab8cd67ba3851f19d823cb0b588c4b', 'sample_003': '10a7c5addeffe06ba7bf29c74ec26bbc8c3fbc3a5bd0bb303118a8e8f652d63a', 'sample_004': 'e11551fbde9254215d8c166efefc467294c7e924179d941cf8fac9a6f90a5cd1'}
    if SAMPLE_NAME not in SAMPLE_HASHES: raise ValueError("Unknown sample")
    path = PACKAGE_DIR/(SAMPLE_NAME+".zip") if ON_COLAB else PROJECT/"outputs/samples/baseline_demo_v1"/(SAMPLE_NAME+".zip")
    SAMPLE_ROOT = unpack(path, SAMPLE_HASHES[SAMPLE_NAME])
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    if INSTALL_DEPS:
        command([sys.executable, "-m", "pip", "install", "-r", CODE_ROOT/"src/requirements.txt"])
    setup = "from brain_vlm.runtime import prepare_environment; prepare_environment("+repr(str(ENV_ROOT))+", 'medgemma', "+repr(INSTALL_DEPS)+", log_dir="+repr(str(RUN_ROOT/"logs"))+")"
    command([sys.executable, "-c", setup])
    MEDGEMMA_PYTHON = ENV_ROOT/("Scripts/python.exe" if os.name == "nt" else "bin/python")
    if ON_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
            if token: ENV["HF_TOKEN"] = token
        except Exception: print("HF_TOKEN unavailable; authorized cache/login may be reused.")

def stage(action, python):
    # The saved logger streams/redacts output and emits a heartbeat during long model calls.
    log = RUN_ROOT/"logs"/(action+"_"+datetime.now(timezone.utc).strftime("%H%M%S_%f")+".log")
    script = "from brain_vlm.runtime import _run_logged; _run_logged("+repr([str(python), "-m", "brain_vlm", action, str(CONFIG_PATH)])+", "+repr(str(log))+", "+repr(action)+", stream=True)"
    command([sys.executable, "-c", script])
print("Source ready; model weights not loaded by setup.")

## 3 · Chuẩn bị input · CPU

Render PNGs và lưu prompt/claims. Không dùng source masks/evaluation labels, không load model.
Khác sequence/context vẫn vào ảnh, nhưng quyền sửa hiện chỉ primary-finding T2 direction.

In [ ]:
if MODE == "run":
    config = {"version":"pretrained_medgemma_signal_v1", "sample_root":str(SAMPLE_ROOT), "output_root":str(RUN_ROOT),
        "variants":DRAFT_VARIANTS, "slices_per_sequence":SLICES_PER_SEQUENCE,
        "models":{"assessor":{"path":str(MODEL_ROOT), "repo_id":"google/medgemma-1.5-4b-it",
            "revision":"91850547d9f0b2fdd21aa7c5f4f3d1a8a52c243b",
            "config_sha256":"300c724c2c1fcdea39f1e21865cb1b14a605f1e3bb5ef50550faaa48da944fc8"}},
        "device":"cuda", "max_images":16, "max_new_tokens":768, "download_missing":DOWNLOAD_MISSING}
    CONFIG_PATH = RUN_ROOT/"config.json"
    if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text()) != config: raise ValueError("Changed config: use a new run")
    CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
    stage("prepare", sys.executable)
    prepared = json.loads((RUN_ROOT/"prepared.json").read_text())
    print("Actual PNGs:", prepared["image_count"], "| unique observation jobs:", prepared["observation_jobs"])
    for variant, row in prepared["requests"].items():
        request = json.loads((RUN_ROOT/row["path"]).read_text())
        print(variant, "|", row["origin"], "|", request["draft_text"])
else: print("Read mode: using saved inputs.")

## 4 · Load MedGemma và inference · GPU

Đây là cell tải model/forward thật. Call cache hợp lệ thì không load lại. Failed/interrupted calls được giữ để đọc,
không tự retry nhiều lần. Nếu chưa có đủ image evidence, output có thể defer; không ép sửa.

In [ ]:
if MODE == "run":
    stage("infer", MEDGEMMA_PYTHON)
    path = RUN_ROOT/"model_load.json"
    if path.exists():
        loaded = json.loads(path.read_text())
        print("Saved model-load record:", loaded["model_loaded"], "| parameters:", loaded["model_metadata"].get("parameter_count"))
        print("Weights bytes:", loaded["snapshot"].get("weight_file_bytes"))
    else: print("No mapped job: no model load. Inspect requests.")
else: print("Read mode: no inference or GPU.")

## 5 · Quyết định và exact edits · CPU

In [ ]:
if MODE == "run":
    stage("finalize", sys.executable)
    summary = json.loads((RUN_ROOT/"summary.json").read_text())
    print(json.dumps(summary, indent=2, ensure_ascii=False))
else: print("Read mode: saved assessments and text remain unchanged.")

## 6 · Đọc kết quả theo luồng · không model

Input PNGs → draft/claims → raw model tensors/output/resources → citations → gate → edits → unresolved.
Review mới đặt ngoài raw run. JSON/schema/contract thành công chưa là y khoa đúng. Samples có lỗi nhân tạo;
baseline này chưa có independent clinical benchmark. Không cần chạy lại GPU để mở presenter sau disconnect.

In [ ]:
if not (RUN_ROOT/"summary.json").exists(): raise FileNotFoundError("No finalized run: inspect logs/observations first")
REVIEW_ROOT = RUN_ROOT.parent/(RUN_ROOT.name+"_review_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
command([sys.executable, "-m", "brain_vlm", "present", RUN_ROOT, "--output", REVIEW_ROOT])
page = REVIEW_ROOT/"presentation.html"
display(HTML(page.read_text(encoding="utf-8")))
print("Raw run:", RUN_ROOT)
print("Presenter:", page)
print("Share the raw run folder/ZIP including PNGs, observations, model metadata and study results.")